# 101 --- Getting Started with ThetaDataDx

This notebook walks through the basics of the `thetadatadx` Python SDK:
installing the package, authenticating, fetching stock and option data,
and computing option Greeks --- all powered by a native Rust backend.

**Prerequisites:** A ThetaData account with an active subscription.

## 1. Installation

Install the SDK with pandas support. The package ships a pre-compiled
Rust binary via PyO3 --- no JVM required.

In [ ]:
# Uncomment to install (run once)
# !pip install thetadatadx[pandas]

## 2. Connect with Credentials

Create a `Credentials` object with your ThetaData email and password,
then connect a `ThetaDataDx` client to the production servers.

In [ ]:
from thetadatadx import Credentials, Config, ThetaDataDx, to_dataframe

# Option A: inline credentials
creds = Credentials("your-email@example.com", "your-password")

# Option B: load from a file (line 1 = email, line 2 = password)
# creds = Credentials.from_file("creds.txt")

config = Config.production()
tdx = ThetaDataDx(creds, config)
print("Connected!", tdx)

## 3. List Stock Symbols

Retrieve the full universe of available stock tickers.

In [ ]:
symbols = tdx.stock_list_symbols()
print(f"Total symbols available: {len(symbols)}")
print(f"First 20: {symbols[:20]}")

## 4. Fetch AAPL End-of-Day Data as a DataFrame

`stock_history_eod` returns a list of tick dicts. Convert to a pandas
DataFrame with `to_dataframe()`.

In [ ]:
import pandas as pd

eod_ticks = tdx.stock_history_eod("AAPL", "20240101", "20240401")
df_eod = to_dataframe(eod_ticks)

print(f"Rows: {len(df_eod)}")
df_eod.head(10)

## 5. Fetch 1-Minute OHLC Bars

Intraday bars for a single trading day. The `interval` argument is
the bar width in milliseconds: `"60000"` = 1 minute.

In [ ]:
ohlc_ticks = tdx.stock_history_ohlc("AAPL", "20240315", "60000")
df_ohlc = to_dataframe(ohlc_ticks)

print(f"1-min bars on 2024-03-15: {len(df_ohlc)}")
df_ohlc.head(10)

## 6. List SPY Option Expirations and Strikes

Options data starts with listing available expirations, then
the strikes for a given expiration.

In [ ]:
# All available expirations for SPY
expirations = tdx.option_list_expirations("SPY")
print(f"Total expirations: {len(expirations)}")
print(f"Next 5: {expirations[:5]}")

# Strikes for the nearest expiration
nearest_exp = expirations[0]
strikes = tdx.option_list_strikes("SPY", nearest_exp)
print(f"\nStrikes for {nearest_exp}: {len(strikes)} total")
print(f"Range: {strikes[0]} -- {strikes[-1]}")
print(f"Sample: {strikes[len(strikes)//2 - 3 : len(strikes)//2 + 3]}")

## 7. Greeks Calculator

The SDK includes a built-in Black-Scholes Greeks calculator written
in Rust. No external dependencies needed --- just pass the inputs.

- `all_greeks()` returns IV + all Greeks (first, second, third order)
- `implied_volatility()` returns just the IV and the numerical error

In [ ]:
from thetadatadx import all_greeks, implied_volatility

# SPY ATM call: spot=$540, strike=$540, r=5%, div yield=1.3%, 30 DTE, mid=$9.80
greeks = all_greeks(
    spot=540.0,
    strike=540.0,
    rate=0.05,
    div_yield=0.013,
    tte=30.0 / 365.0,
    option_price=9.80,
    is_call=True,
)

print("=== SPY 540C  30 DTE ===")
print(f"  IV:        {greeks['iv']:.4f}  (error: {greeks['iv_error']:.2e})")
print(f"  Delta:     {greeks['delta']:.4f}")
print(f"  Gamma:     {greeks['gamma']:.6f}")
print(f"  Theta:     {greeks['theta']:.4f}")
print(f"  Vega:      {greeks['vega']:.4f}")
print(f"  Rho:       {greeks['rho']:.4f}")
print(f"  Vanna:     {greeks['vanna']:.6f}")
print(f"  Charm:     {greeks['charm']:.6f}")
print(f"  Vomma:     {greeks['vomma']:.6f}")
print(f"  Speed:     {greeks['speed']:.8f}")
print(f"  Zomma:     {greeks['zomma']:.8f}")
print(f"  Color:     {greeks['color']:.8f}")
print(f"  Ultima:    {greeks['ultima']:.6f}")

In [ ]:
# Standalone implied volatility (faster if you only need IV)
iv, iv_err = implied_volatility(
    spot=540.0,
    strike=540.0,
    rate=0.05,
    div_yield=0.013,
    tte=30.0 / 365.0,
    option_price=9.80,
    is_call=True,
)
print(f"IV = {iv:.4f}  (numerical error: {iv_err:.2e})")

---

**Next:** [102 --- Historical Data Analysis](102_historical_analysis.ipynb)